# Day 14：第一阶段里程碑项目一 —— 《命令行多轮对话 AI 助手》

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 14 天！今天我们将前 13 天构建的消息模型、HTTP 客户端、重试装饰器与分层异常体系正式集成为 `MultiTurnAssistant`，完成项目一的端到端交付与自动化验证。

## 1. 环境初始化与导入 `project_iterations/day_14` 模块
在无 `OPENAI_API_KEY` 的环境下，`LLMClient` 会自动启用 `httpx.MockTransport` 离线模拟通道，确保 Notebook 在任何机器上都可无阻塞运行。

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# 自动定位仓库根目录下的 project_iterations/day_14
repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day14_dir = repo_root / "project_iterations" / "day_14"
if str(day14_dir) not in sys.path:
    sys.path.insert(0, str(day14_dir))

# 确保演示使用离线 Mock 模式
os.environ.pop("OPENAI_API_KEY", None)

from zhishu_agent import MultiTurnAssistant, LLMClient, CommandError, LLMAPIError
import httpx

assistant = MultiTurnAssistant(system_prompt="你是智枢平台项目一的 AI 研发助手。")
print(f"初始化成功！当前模式: {'Mock 离线模拟' if assistant.client.is_mock else 'Live API'}")
print(f"初始消息条数: {len(assistant.messages)} -> {assistant.messages[0]}")

初始化成功！当前模式: Mock 离线模拟
初始消息条数: 1 -> [system] 你是智枢平台项目一的 AI 研发助手。


## 2. 端到端多轮对话记忆演示
连续发送多轮提问，观察 `MultiTurnAssistant` 如何维护 `messages` 列表并在第 2 轮中记住第 1 轮的用户信息。

In [2]:
for user_text in [
    "你好，我是工程师陈默，正在交付项目一。",
    "你还记得我是谁、在做什么吗？",
]:
    cont, reply = assistant.process_input(user_text)
    print(f"用户: {user_text}")
    print(f"助手: {reply}\n")

print(f"当前上下文消息总数: {len(assistant.messages)} 条")

用户: 你好，我是工程师陈默，正在交付项目一。
助手: [Mock-gpt-4o-mini | 第1轮] 已收到你的消息：「你好，我是工程师陈默，正在交付项目一。」。

用户: 你还记得我是谁、在做什么吗？
助手: [Mock-gpt-4o-mini | 第2轮] 已收到你的消息：「你还记得我是谁、在做什么吗？」。（我记得你在前文提到过：你好，我是工程师陈默，正在交付项目一。）

当前上下文消息总数: 5 条


## 3. 斜杠命令实战：`/stats`、`/save`、`/clear`、`/load` 与 `/exit`
使用临时目录验证会话保存、清空（保留 System Prompt）、加载恢复与统计输出。

In [3]:
# 1) 查看 /stats
_, stats_out = assistant.process_input("/stats")
print(stats_out)
print("-" * 50)

# 2) /save 保存至临时 JSON，再 /clear 清空历史，最后 /load 恢复
with tempfile.TemporaryDirectory() as tmpdir:
    session_path = Path(tmpdir) / "day14_session.json"
    _, save_out = assistant.process_input(f"/save {session_path}")
    print(save_out)

    _, clear_out = assistant.process_input("/clear")
    print(clear_out)
    print(f"清空后消息数: {len(assistant.messages)}（仅保留 {assistant.messages[0].role}）")

    _, load_out = assistant.process_input(f"/load {session_path}")
    print(load_out)
    print(f"恢复后消息数: {len(assistant.messages)}, 轮数: {assistant.turns}")

# 3) /exit 指令返回退出信号
cont, exit_out = assistant.process_input("/exit")
print(f"/exit -> should_continue={cont}, message={exit_out}")

📊 会话统计摘要
  - 模型与模式 : gpt-4o-mini (Mock 离线模拟)
  - 对话轮数   : 2 轮
  - 上下文消息 : 5 条
  - Token 消耗 : prompt=70, completion=68, total=138
--------------------------------------------------
💾 会话已保存至 /tmp/tmp6gp05zxu/day14_session.json（共 5 条消息）。
✅ 对话历史已清空（已保留 System Prompt）。
清空后消息数: 1（仅保留 system）
📂 已从 /tmp/tmp6gp05zxu/day14_session.json 加载会话（共 5 条消息，2 轮对话）。
恢复后消息数: 5, 轮数: 2
/exit -> should_continue=False, message=👋 再见！感谢使用智枢智能体平台。


## 4. 异常重试与历史回滚验证
模拟一次“第 1 次 429 限流、第 2 次成功”的瞬时故障恢复，以及一次“401 不可重试错误”触发的上下文回滚。

In [4]:
attempts = {"n": 0}

def flaky_handler(request: httpx.Request) -> httpx.Response:
    attempts["n"] += 1
    if attempts["n"] == 1:
        return httpx.Response(429, json={"error": "rate limit"})
    return httpx.Response(
        200,
        json={
            "model": "gpt-4o-mini",
            "choices": [{"message": {"role": "assistant", "content": "429 重试后成功恢复！"}}],
            "usage": {"prompt_tokens": 10, "completion_tokens": 8, "total_tokens": 18},
        },
    )

resilient_bot = MultiTurnAssistant(
    client=LLMClient(api_key="demo", max_retries=2, retry_delay=0.0, transport=httpx.MockTransport(flaky_handler))
)
print("瞬时 429 测试:", resilient_bot.ask("测试重试"))
print(f"总请求次数: {attempts['n']}")

# 401 不可重试错误回滚测试
auth_fail_bot = MultiTurnAssistant(
    client=LLMClient(api_key="bad", max_retries=2, retry_delay=0.0, transport=httpx.MockTransport(lambda r: httpx.Response(401, text="Unauthorized")))
)
try:
    auth_fail_bot.ask("这条失败消息应被回滚")
except LLMAPIError as exc:
    print(f"捕获预期异常: {exc}")
    print(f"回滚后消息数: {len(auth_fail_bot.messages)}（仅剩 System Prompt）")

瞬时 429 测试: 429 重试后成功恢复！
总请求次数: 2
捕获预期异常: [LLM_API_ERROR] 大模型请求被拒绝 (HTTP 401): Unauthorized
回滚后消息数: 1（仅剩 System Prompt）
